## Installation (execute only if running on a cloud platform!)¶

In [ ]:
# -- Use the following for Google Colab
# ! pip install -q 'lalsuite==7.11' 'PyCBC==2.0.5' 

## Dowloading GW events

Below we will use the PyCBC functions to download data correspondant to GW150914. PyCBC is build on two types of classes: timeseries and frequencyseries classes. These classes contains several methods that are useful for GW data analysis

In [ ]:
%matplotlib inline

# Read in the data around GW150914
from pycbc.catalog import Merger
import matplotlib.pyplot as plt

m = Merger('GW150914')

data = {}
for ifo in ['H1', 'L1']:
    data[ifo] = m.strain(ifo)

In [ ]:
for ifo in data:
    plt.plot(data[ifo].sample_times, data[ifo], label=ifo)

plt.ylabel('Strain')
plt.xlabel('GPS Time (s)')
plt.legend()
plt.show()

In [ ]:
for ifo in data:
    # Here we are cutting  just visualizing the data withing 1 second from the merger time
    zoom = data[ifo].time_slice(m.time - 0.5, m.time + 0.5)
    plt.plot(zoom.sample_times, zoom, label=ifo)

plt.legend()
plt.show()

As you can see, the data does not look like the cool plots that we show around. Before obtaining those plots, we need to further clean the data. GW data is strongly contaminated by low-frequency seismic noise. So we need to cut off all the frequency components which are below 15 Hz. We need to pass an high pass filter.

In [ ]:
for ifo in data:
    high_data = data[ifo].highpass_fir(15, 512) # Highpass point is 15 Hz
    zoom = high_data.time_slice(m.time - 0.5, m.time + 0.5)
    plt.plot(zoom.sample_times, zoom, label=ifo)

plt.legend()
plt.show()

The situation now it's better, as you can see, we have removed from L1 the continuous background. Now the time series is still centered around 0.
However, if you remember from the lecture, the noise in some frequency bands is stronger than in the others. So what we are observing is not white gaussian noise, but instead coulered noise plus some noise lines. We need to estimate the power spectral density and renormalize our data

In [ ]:
for ifo in data:
    # This estimates the PSD by sub-dividing the data into overlapping
    # 4s long segments
    psd = data[ifo].psd(4)
    
    # Note that the psd is a FrequencySeries!
    plt.loglog(psd.sample_frequencies, psd, label=ifo)
    
plt.ylabel('Strain$^2$ / Hz') # Note this is ^2! compared to notes
plt.xlabel('Frequency (Hz)')
plt.grid()
plt.xlim(10, 1500)
plt.legend()
plt.show()

In [ ]:
# Whiten the data
whitened = {}

for ifo in data:
    whitened[ifo] = data[ifo].whiten(4, 4)

    zoom = whitened[ifo].time_slice(m.time - 0.5, m.time + 0.5)
    plt.plot(zoom.sample_times, zoom, label=ifo)

plt.ylabel('Whitened Strain')
plt.xlabel('Time (s)')
plt.legend()
plt.show()

The data is now "whitened", i.e. the noise is white and gaussian (with a flat spectrum). If there is a strong singal, now we might be able to see it. We want to zoom now!

In [ ]:
for ifo in whitened:
    # Apply a highpass filter (at 30 Hz) followed by an lowpass filter (at 250 Hz)
    bpsd = whitened[ifo].highpass_fir(30, 512).lowpass_fir(250, 512)
    
    zoom = bpsd.time_slice(m.time - 0.2, m.time + 0.1)
    plt.plot(zoom.sample_times, zoom, label=ifo)

plt.grid()
plt.legend()
plt.show()


Here is the inspiral-merger-ringdown!

Next, we apply a Q-transform to look at the chirp in frquency. Q is a quality factor which roughly tells how many cycles you have within some window. The idea is that "Fourier transforms to short windowed segments of the original time series to create a uniform time-frequency map with a time resolution and frequency resolution that depends upon the window’s duration. However, it is impossible to pick a window duration appropriate for all frequencies. Low frequency signals require long duration windows to accumulate suﬃcient frequency information, while high frequency signals are better isolated in time using short duration windows." - citing https://arxiv.org/pdf/gr-qc/0412119 
That is why one used the Q-transform, which keeps Q fixed and lets the window duration scale with frequency.

In [ ]:
for ifo in whitened:
    # We'll choose a tighter zoom here.
    zoom = whitened[ifo].time_slice(m.time - 5, m.time + 5)
    times, freqs, power = zoom.qtransform(qrange=(8, 8), # set the range of Q values searched - higher Q: narrow band/long duration signal, lower Q: broader band/short duration signal
                                                frange=(20, 512)) #frequency band to be searched
    plt.figure(figsize=[15, 3])
    plt.pcolormesh(times, freqs, power**0.5)
    plt.xlim(m.time - 0.5, m.time + 0.3)
    plt.title(ifo)
    plt.yscale('log')
    plt.show()

And the chirp signal!

## Matched Filtering

So far we've found the signal "by eye" using whitening and the Q-transform. Real GW searches instead use **matched filtering**: correlating the noisy data against a bank of theoretical waveform templates to find the exact merger time and quantify how significant a candidate is.

Below we build a template waveform with parameters close to GW150914's and match-filter it against the H1 strain.

In [ ]:
from pycbc.waveform import get_td_waveform
from pycbc.filter import matched_filter
from pycbc.psd import interpolate, inverse_spectrum_truncation

ifo = 'H1'

# Estimate the PSD from the full data segment (not just around the merger)
psd = data[ifo].psd(4)
psd = interpolate(psd, data[ifo].delta_f)
psd = inverse_spectrum_truncation(psd, int(4 * data[ifo].sample_rate),
                                   low_frequency_cutoff=15.0)

# Generate a template waveform close to GW150914's parameters
hp, _ = get_td_waveform(approximant="IMRPhenomD",
                         mass1=36, mass2=29,
                         delta_t=data[ifo].delta_t,
                         f_lower=20)

# Resize/align the template to match the data length
hp.resize(len(data[ifo]))
template = hp.cyclic_time_shift(hp.start_time)

# Matched filter: correlate the template against the data, weighted by the PSD
snr = matched_filter(template, data[ifo], psd=psd, low_frequency_cutoff=20)

# Remove edge effects introduced by filtering
snr = snr.crop(4, 4)

plt.figure(figsize=(10, 4))
plt.plot(snr.sample_times, abs(snr))
plt.ylabel('Signal-to-noise ratio')
plt.xlabel('GPS Time (s)')
plt.title(f'Matched filter SNR in {ifo}')
plt.show()

peak = abs(snr).numpy().argmax()
snrp = snr[peak]
time = snr.sample_times[peak]
print(f"Peak SNR = {abs(snrp):.2f} at GPS time {time:.3f} (catalog merger time: {m.time:.3f})")

**Task:** After running the cells above, try changing `mass1` and `mass2` to values very different from GW150914's true masses. What happens to the SNR peak, and why?

## Challenge GW170814

In [ ]:
%matplotlib inline

# Read in the data around merger
from pycbc.catalog import Merger
import pylab

m = Merger('GW170814')

data = {}
for ifo in ['H1', 'L1','V1']:
    data[ifo] = m.strain(ifo)

In [ ]:
for ifo in data:
    # This estimates the PSD by sub-dividing the data into overlapping
    # 4s long segments. (See Welch's method)
    psd = data[ifo].psd(4)
    
    # Note that the psd is a FrequencySeries!
    pylab.loglog(psd.sample_frequencies, psd, label=ifo)
    
pylab.ylabel('$Strain^2 / Hz$')
pylab.xlabel('Frequency (Hz)')
pylab.grid()
pylab.legend()
pylab.xlim(10, 1100)
pylab.show()

In [ ]:
# Whiten the data
whitened = {}

for ifo in data:
    # This produces a whitened set.
    # This works by estimating the power spectral density from the
    # data and then flattening the frequency response.
    # (1) The first option sets the duration in seconds of each
    #     sample of the data used as part of the PSD estimate.
    # (2) The second option sets the duration of the filter to apply
    whitened[ifo] = data[ifo].whiten(4, 4)

    zoom = whitened[ifo].time_slice(m.time - 0.5, m.time + 0.5)
    pylab.plot(zoom.sample_times, zoom, label=ifo)

pylab.ylabel('Whitened Strain')
pylab.xlabel('Time (s)')
pylab.legend()
pylab.show()

In [ ]:
#pylab.figure(figsize=[15, 3])
for ifo in whitened:
    # Apply a highpass filter (at 30 Hz) followed by an lowpass filter (at 250 Hz)
    bpsd = whitened[ifo].highpass_fir(30, 512).lowpass_fir(250, 512)
    
    # We'll choose a tighter zoom here.
    zoom = bpsd.time_slice(m.time - .1, m.time + .1)
    pylab.plot(zoom.sample_times, zoom, label=ifo)

pylab.grid()
pylab.legend()
pylab.show()

In [ ]:
for ifo in whitened:
    # We'll choose a tighter zoom here.
    zoom = whitened[ifo].time_slice(m.time - 5, m.time + 5)
                    
    times, freqs, power = zoom.qtransform(qrange=(10, 12),frange=(20, 512))
    pylab.figure(figsize=[15, 3])
    pylab.pcolormesh(times, freqs, power**0.5)
    pylab.xlim(m.time - 0.5, m.time + 0.3)
    pylab.title(ifo)
    pylab.yscale('log')
    pylab.show()

This was the first 3 detector detection! Only 3 days before GW170817...

**Task:** why do you think the signal looks so different in Virgo compared to LIGO?
Estimate qualitatively how much V1 contributes to the network SNR for GW170814 compared to H1 and L1. (Hint: matched-filter SNR scales with how much power the template has where the noise is *low* — a noisier detector contributes less, even with the same physical signal passing through it.)

## Challenge: GW170817 — a binary neutron star merger

GW150914 and GW170814 are both binary black hole mergers with total masses of tens of solar masses — the signal only lasts a fraction of a second in the LIGO/Virgo band. GW170817 is different: it's the merger of two **neutron stars**, with a much lower total mass (~2.8 solar masses).

GW170817 is also the only confirmed multi-messenger compact binary merger to date: it was accompanied by a short gamma-ray burst (GRB 170817A) about 1.7 seconds after merger, and by a kilonova (AT2017gfo) found in follow-up imaging — the event that confirmed neutron star mergers as a site of r-process nucleosynthesis.

**Before running anything:** think about the earlier cells. They all used a `time_slice` window of about ±0.5 seconds around the merger. Given the physics above, do you expect that window to work for GW170817? Try the naive version below and see what happens, then fix it.

In [ ]:
m17 = Merger('GW170817')

data17 = {}
for ifo in ['H1', 'L1']:
    data17[ifo] = m17.strain(ifo, duration=4096)

In [ ]:
for ifo in data17:
    zoom = data17[ifo].time_slice(m17.time - 0.5, m17.time + 0.5)
    plt.plot(zoom.sample_times, zoom, label=ifo)
plt.legend()
plt.title("±0.5s around merger")
plt.show()

**Tasks:**

1. Roughly how many seconds before merger can you trace the chirp track in the Q-transform, compared to GW150914's fraction of a second?
2. Using the df/dt relation (below, eq. 5.2.30 from the notes), try roughly estimating GW170817's chirp mass from the track. 

$$\dot{f}=\frac{96}{5} \pi^{8/3} \big( \frac{G{M}_c}{c^3}\big)^{5/3} f^{11/3} $$

## Bonus challenge: GW230529 — neutron star or black hole?

GW230529 is one of the most interesting events from the O4 observing run: a compact binary merger where the *secondary* object has a mass in the "lower mass gap" (~2.5–4.5 M☉) — right between the heaviest known neutron stars and the lightest known black holes.

Unlike GW170817, GW230529 was not accompanied by a confirmed electromagnetic counterpart.

**Task:**
Run the same highpass → whiten → Q-transform pipeline and compare the chirp to GW170817's.

In [ ]:
# Attempt via PyCBC's built-in catalog 
try:
    m230529 = Merger('GW230529_181500')
    print("Loaded via PyCBC catalog.")
except Exception as e:
    print(f"Not found in the built-in catalog ({e}).")